**0. Imports:**

In [ ]:
import earthcarekit as eck
eck.set_config("/usr/people/raucher/Documents/Config_ECK/default_config.toml")
import xarray as xr
xr.set_options(display_expand_data_vars=True, display_max_rows=500, display_max_items=10000)
import numpy as np
import pandas as pd
from IPython.display import display
from pathlib import Path
from datetime import datetime, timedelta, date
import matplotlib.pyplot as plt
import os
from scipy.interpolate import interp1d
import sys, shlex, shutil

**0b. Remote ZIP Helpers**

In [ ]:
MY_SUBPROCESS_FILE = Path("/usr/people/raucher/Documents/Coding1/Gerd-Jan/OneDrive_1_24-2-2026")
if str(MY_SUBPROCESS_FILE) not in sys.path:
    sys.path.insert(0, str(MY_SUBPROCESS_FILE))
from my_subprocess import run_shell_cmd_and_communicate, print_shell_output

def _to_date(v):
    if isinstance(v, datetime): return v.date()
    if isinstance(v, date):     return v
    if isinstance(v, str):      return datetime.strptime(v, "%Y-%m-%d").date()
    raise TypeError(f"Unsupported type: {type(v)}")

def run_cmd_checked(cmd, verbose=False):
    lines_out, lines_err, rc = run_shell_cmd_and_communicate(cmd, verbose=verbose)
    if rc != 0:
        print_shell_output(lines_out, lines_err, prefix="[shell] ")
        raise RuntimeError(f"Command failed (exit {rc}): {cmd}")
    return lines_out, lines_err

def discover_remote_zip_files(remote_product_root, start_date, end_date):
    root, start, end = Path(remote_product_root), _to_date(start_date), _to_date(end_date)
    if end < start: raise ValueError("end_date must be >= start_date")
    zips, day = [], start
    while day <= end:
        d = root / day.strftime("%Y") / day.strftime("%m") / day.strftime("%d")
        if d.exists():
            zips.extend(sorted(d.glob("*.ZIP")))
            zips.extend(sorted(d.glob("*.zip")))
        day += timedelta(days=1)
    return sorted(dict.fromkeys(zips))

def stage_zip_and_extract(src_zip, local_stage_root):
    local_stage_root.mkdir(parents=True, exist_ok=True)
    local_zip   = local_stage_root / src_zip.name
    extract_dir = local_stage_root / src_zip.stem
    if local_zip.exists():   local_zip.unlink()
    if extract_dir.exists(): shutil.rmtree(extract_dir)
    run_cmd_checked(f"cp {shlex.quote(str(src_zip))} {shlex.quote(str(local_zip))}")
    run_cmd_checked(f"unzip -oq {shlex.quote(str(local_zip))} -d {shlex.quote(str(extract_dir))}")
    h5_files = sorted(extract_dir.rglob("*.h5"))
    if not h5_files: raise FileNotFoundError(f"No .h5 after extracting: {src_zip}")
    return local_zip, extract_dir, h5_files

def cleanup_staged_data(local_zip, extract_dir):
    if extract_dir is not None and extract_dir.exists(): shutil.rmtree(extract_dir)
    if local_zip   is not None and local_zip.exists():   local_zip.unlink()

**1. Config**

In [ ]:
REMOTE_PRODUCT_ROOT = Path("/net/pc230016/nobackup_1/users/zadelhof/EarthCARE_DATA/L2/ATL_EBD_2A")

LOCAL_STAGE_ROOT = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/temp_ZIPs_EBD")
LOCAL_STAGE_ROOT.mkdir(parents=True, exist_ok=True)

START_DATE = "2025-12-01"
END_DATE   = "2025-12-31"

OUTPUT_ROOT = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/ATC_output")

# Variables to load and grid (main + error pairs)
DATA_VARS = [
    "lidar_ratio_355nm",
    "particle_backscatter_coefficient_355nm",
    "particle_extinction_coefficient_355nm",
    "particle_linear_depol_ratio_355nm",
    "rayleigh_attenuated_backscatter_355nm",
    "total_attenuated_backscatter_355nm",
    "mie_total_attenuated_backscatter_355nm",
]
ERROR_VARS = [v + "_error" for v in DATA_VARS]
VAR_PAIRS  = list(zip(DATA_VARS, ERROR_VARS))

# Short names used in output NetCDF variable names
SHORT_NAMES = {
    "lidar_ratio_355nm":                       "lidar_ratio",
    "particle_backscatter_coefficient_355nm":  "backscatter",
    "particle_extinction_coefficient_355nm":   "extinction",
    "particle_linear_depol_ratio_355nm":       "depol_ratio",
    "rayleigh_attenuated_backscatter_355nm":   "rayleigh_bsc",
    "total_attenuated_backscatter_355nm":      "total_bsc",
    "mie_total_attenuated_backscatter_355nm":  "mie_total_bsc",
}

LAT        = "latitude"
LON        = "longitude"
HEIGHT_VAR = "height"

GRID_RES_DEG         = 1.0
MAX_HEIGHT_M         = 20_000.0
MIN_SAMPLES_PER_CELL = 10

# --- ATL_TC ice mask ---
REMOTE_TC_ROOT      = Path("/net/pc230016/nobackup_1/users/zadelhof/EarthCARE_DATA/L2/ATL_TC__2A")
LOCAL_STAGE_TC_ROOT = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/temp_ZIPs_TC_for_EBD")
LOCAL_STAGE_TC_ROOT.mkdir(parents=True, exist_ok=True)
TC_CLASS_VAR     = "classification"
TC_ICE_CODE      = 3       # Ice Cloud
TC_EXCLUDE_CODES = [-2, -1]  # ground, noise

start_dt = datetime.strptime(START_DATE, "%Y-%m-%d").date()
end_dt   = datetime.strptime(END_DATE,   "%Y-%m-%d").date()
if end_dt < start_dt:
    raise ValueError("END_DATE must be >= START_DATE")
if not REMOTE_PRODUCT_ROOT.exists():
    raise FileNotFoundError(f"Remote path not mounted/reachable: {REMOTE_PRODUCT_ROOT}")

print("Config loaded")
print(f"Product:     ATL_EBD_2A")
print(f"Remote root: {REMOTE_PRODUCT_ROOT}")
print(f"Stage root:  {LOCAL_STAGE_ROOT}")
print(f"Date range:  {start_dt} to {end_dt}")
print(f"DATA_VARS ({len(DATA_VARS)}): {DATA_VARS}")
print(f"MAX_HEIGHT_M={MAX_HEIGHT_M:.0f} m  GRID_RES_DEG={GRID_RES_DEG}")
print(f"TC mask:     ATL_TC__2A (ice code={TC_ICE_CODE}, exclude={TC_EXCLUDE_CODES})")

**2. File Discovery**

In [ ]:
import re as _re

zip_paths = discover_remote_zip_files(REMOTE_PRODUCT_ROOT, start_dt, end_dt)

print(f"Discovered {len(zip_paths)} ZIP files  |  Range: {start_dt} -> {end_dt}")
for p in zip_paths[:3]:
    print("  -", p)
if len(zip_paths) > 3:
    print(f"  ... and {len(zip_paths)-3} more")

if len(zip_paths) == 0:
    raise FileNotFoundError("No ZIP files found for this date range/product.")

# Discover ATL_TC ZIPs for ice masking and build orbit-key lookup
tc_zip_paths = discover_remote_zip_files(REMOTE_TC_ROOT, start_dt, end_dt)

def _orbit_key(p):
    """Extract orbit+frame identifier (e.g. '08575A') from a ZIP or H5 path."""
    m = _re.search(r'_(\d{5}[A-Z])$', Path(p).stem)
    return m.group(1) if m else None

tc_zip_by_orbit = {_orbit_key(p): p for p in tc_zip_paths if _orbit_key(p)}
print(f"\nTC mask: {len(tc_zip_paths)} ZIPs discovered, {len(tc_zip_by_orbit)} matched by orbit key")

**3. Single-File Inspection**

In [ ]:
local_zip = None
extract_dir = None
try:
    local_zip, extract_dir, staged_h5 = stage_zip_and_extract(zip_paths[0], LOCAL_STAGE_ROOT)
    fp0 = str(staged_h5[0])
    print("Inspecting:", fp0)

    with eck.read_product(fp0) as ds0:
        print("\nDataset summary:")
        display(ds0)

        all_required = DATA_VARS + ERROR_VARS + [HEIGHT_VAR, LAT, LON]
        print("\nRequired variable check:")
        for v in all_required:
            status = "OK" if v in ds0.data_vars else "MISSING"
            print(f"  {status}  {v}")

        print("\nVariable statistics (first file):")
        for v in DATA_VARS:
            arr   = ds0[v].values.astype(float)
            valid = np.isfinite(arr)
            if valid.any():
                print(f"  {SHORT_NAMES[v]:15s}: min={np.nanmin(arr):.3g}  "
                      f"max={np.nanmax(arr):.3g}  "
                      f"valid={100*valid.mean():.1f}%  "
                      f"units={ds0[v].attrs.get('units','?')}")
            else:
                print(f"  {SHORT_NAMES[v]:15s}: ALL NaN")
finally:
    cleanup_staged_data(local_zip, extract_dir)

**4. Grid Setup**

In [ ]:
lat_bins = np.arange(-90.0, 90.0 + GRID_RES_DEG, GRID_RES_DEG)
lon_bins = np.arange(-180.0, 180.0 + GRID_RES_DEG, GRID_RES_DEG)

lat_centers = 0.5 * (lat_bins[:-1] + lat_bins[1:])
lon_centers = 0.5 * (lon_bins[:-1] + lon_bins[1:])

n_lat = len(lat_centers)
n_lon = len(lon_centers)

# Extract vertical grid from first file, capped at MAX_HEIGHT_M
local_zip = None
extract_dir = None
try:
    local_zip, extract_dir, staged_h5 = stage_zip_and_extract(zip_paths[0], LOCAL_STAGE_ROOT)
    with eck.read_product(str(staged_h5[0])) as ds0:
        _h = ds0[HEIGHT_VAR].values[0, :]
        target_h = _h[_h <= MAX_HEIGHT_M]
finally:
    cleanup_staged_data(local_zip, extract_dir)

n_height = target_h.size

# Global accumulators: per variable, 5 arrays each
# sum, sum_sq, count  -> for mean and std dev of the main variable
# sum_err, count_err  -> for mean of the per-pixel retrieval error
global_acc = {
    v: {k: np.zeros((n_lat, n_lon, n_height), dtype=np.float64)
        for k in ("sum", "sum_sq", "count", "sum_err", "count_err")}
    for v in DATA_VARS
}

print("Grid ready")
print(f"Shape (lat, lon, height): {(n_lat, n_lon, n_height)}")
print(f"Height range: {float(np.nanmin(target_h)):.0f} .. {float(np.nanmax(target_h)):.0f} m")
print(f"Variables to accumulate: {len(DATA_VARS)}")
assert np.nanmax(target_h) <= MAX_HEIGHT_M

**5. One-File Accumulator**

In [ ]:
"""
For each ATL_EBD_2A file, interpolate all DATA_VARS onto the common vertical grid
and accumulate sum, sum-of-squares, and count per (lat, lon, height) cell.
Error variables are accumulated separately (sum_err, count_err) for mean error.

When tc_fp is provided (matching ATL_TC__2A granule), only pixels classified as
ice cloud (TC_ICE_CODE=3) are accumulated; all other pixels are treated as NaN.
"""

def _build_ice_mask(tc_fp, n_obs, target_h):
    """Return bool array (n_obs, n_h): True where ATL_TC classifies as ice cloud."""
    n_h = target_h.size
    ice_mask = np.zeros((n_obs, n_h), dtype=bool)
    with eck.read_product(str(tc_fp)) as ds_tc:
        tc_cls = ds_tc[TC_CLASS_VAR].values.astype(float)
        tc_h   = ds_tc[HEIGHT_VAR].values.astype(float)
    n_tc = tc_cls.shape[0]
    for i in range(min(n_obs, n_tc)):
        h_i   = tc_h[i, :]
        cls_i = tc_cls[i, :]
        valid = np.isfinite(h_i) & np.isfinite(cls_i) & ~np.isin(cls_i, TC_EXCLUDE_CODES)
        if np.sum(valid) < 2:
            continue
        h_v   = h_i[valid]
        ice_v = (cls_i[valid] == TC_ICE_CODE).astype(float)
        order = np.argsort(h_v)
        h_v, ice_v = h_v[order], ice_v[order]
        h_v, idx   = np.unique(h_v, return_index=True)
        ice_v      = ice_v[idx]
        if h_v.size < 2:
            continue
        f = interp1d(h_v, ice_v, kind="nearest", bounds_error=False, fill_value=0.0)
        ice_mask[i, :] = f(target_h).astype(bool)
    return ice_mask


def accumulate_one_file(fp, lat_bins, lon_bins, target_h, tc_fp=None):
    n_lb, n_lonb, n_h = len(lat_bins)-1, len(lon_bins)-1, target_h.size

    acc = {
        v: {k: np.zeros((n_lb, n_lonb, n_h), dtype=np.float64)
            for k in ("sum", "sum_sq", "count", "sum_err", "count_err")}
        for v in DATA_VARS
    }

    with eck.read_product(str(fp)) as ds:
        h   = ds[HEIGHT_VAR].values.astype(float)
        lat = ds[LAT].values
        lon = ds[LON].values
        arrs = {v: ds[v].values.astype(float) for v in DATA_VARS + ERROR_VARS}
        n_obs = h.shape[0]

    # Build ice mask from ATL_TC if available
    ice_mask = _build_ice_mask(tc_fp, n_obs, target_h) if tc_fp is not None else None

    # Per-profile interpolation onto target_h
    interp_d = {v: np.full((n_obs, n_h), np.nan) for v in DATA_VARS}
    interp_e = {v: np.full((n_obs, n_h), np.nan) for v in ERROR_VARS}

    for i in range(n_obs):
        h_i = h[i, :]
        for v_main, v_err in VAR_PAIRS:
            d_i = arrs[v_main][i, :]
            e_i = arrs[v_err][i, :]

            valid = np.isfinite(h_i) & np.isfinite(d_i)
            if np.sum(valid) < 2:
                continue

            h_v = h_i[valid]; d_v = d_i[valid]; e_v = e_i[valid]
            order = np.argsort(h_v)
            h_v = h_v[order]; d_v = d_v[order]; e_v = e_v[order]
            h_v, idx = np.unique(h_v, return_index=True)
            d_v = d_v[idx]; e_v = e_v[idx]
            if h_v.size < 2:
                continue

            interp_d[v_main][i, :] = interp1d(
                h_v, d_v, kind="linear", bounds_error=False, fill_value=np.nan)(target_h)
            interp_e[v_err][i, :]  = interp1d(
                h_v, e_v, kind="linear", bounds_error=False, fill_value=np.nan)(target_h)

        # Apply ice mask: NaN out non-ice pixels for this profile
        if ice_mask is not None:
            not_ice = ~ice_mask[i, :]
            for v_main in DATA_VARS:
                interp_d[v_main][i, not_ice] = np.nan
            for v_err in ERROR_VARS:
                interp_e[v_err][i, not_ice] = np.nan

    lat2d = np.broadcast_to(lat[:, None], (n_obs, n_h))
    lon2d = np.broadcast_to(lon[:, None], (n_obs, n_h))

    for k in range(n_h):
        lat_k = lat2d[:, k]
        lon_k = lon2d[:, k]
        for v_main, v_err in VAR_PAIRS:
            d = interp_d[v_main][:, k]
            e = interp_e[v_err][:, k]

            m = np.isfinite(d) & np.isfinite(lat_k) & np.isfinite(lon_k)
            if np.any(m):
                cnt,  _, _ = np.histogram2d(lat_k[m], lon_k[m], bins=[lat_bins, lon_bins])
                s,    _, _ = np.histogram2d(lat_k[m], lon_k[m], bins=[lat_bins, lon_bins], weights=d[m])
                s2,   _, _ = np.histogram2d(lat_k[m], lon_k[m], bins=[lat_bins, lon_bins], weights=d[m]**2)
                acc[v_main]["count"][:, :, k]  += cnt
                acc[v_main]["sum"][:, :, k]    += s
                acc[v_main]["sum_sq"][:, :, k] += s2

            m_e = m & np.isfinite(e)
            if np.any(m_e):
                ce, _, _ = np.histogram2d(lat_k[m_e], lon_k[m_e], bins=[lat_bins, lon_bins])
                se, _, _ = np.histogram2d(lat_k[m_e], lon_k[m_e], bins=[lat_bins, lon_bins], weights=e[m_e])
                acc[v_main]["count_err"][:, :, k] += ce
                acc[v_main]["sum_err"][:, :, k]   += se

    return acc


def merge_acc(global_acc, file_acc):
    """Add file_acc into global_acc in-place."""
    for v in DATA_VARS:
        for k in ("sum", "sum_sq", "count", "sum_err", "count_err"):
            global_acc[v][k] += file_acc[v][k]


### TEST ###
local_zip = local_tc_zip = None
extract_dir = tc_extract_dir = None
try:
    local_zip, extract_dir, staged_h5 = stage_zip_and_extract(zip_paths[0], LOCAL_STAGE_ROOT)
    orbit   = _orbit_key(zip_paths[0])
    tc_zip  = tc_zip_by_orbit.get(orbit)
    tc_h5   = None
    if tc_zip is not None:
        local_tc_zip, tc_extract_dir, tc_h5_files = stage_zip_and_extract(tc_zip, LOCAL_STAGE_TC_ROOT)
        tc_h5_by_orbit = {_orbit_key(p): p for p in tc_h5_files if _orbit_key(p)}
        tc_h5 = tc_h5_by_orbit.get(orbit)
    test_acc = accumulate_one_file(staged_h5[0], lat_bins, lon_bins, target_h, tc_fp=tc_h5)
    masked = "ice-masked" if tc_h5 is not None else "UNMASKED (no TC file found)"
    print(f"One-file accumulator test ({masked}):")
    for v in DATA_VARS:
        n = int(np.nansum(test_acc[v]["count"]))
        print(f"  {SHORT_NAMES[v]:15s}: {n} valid pixels")
finally:
    cleanup_staged_data(local_zip, extract_dir)
    cleanup_staged_data(local_tc_zip, tc_extract_dir)

**6. Few-File Test**

In [ ]:
N_QUICK_TEST = min(5, len(zip_paths))
test_global  = {
    v: {k: np.zeros((n_lat, n_lon, n_height), dtype=np.float64)
        for k in ("sum", "sum_sq", "count", "sum_err", "count_err")}
    for v in DATA_VARS
}
failed_test = []
processed_h5_test = 0

for src_zip in zip_paths[:N_QUICK_TEST]:
    local_zip = local_tc_zip = None
    extract_dir = tc_extract_dir = None
    try:
        local_zip, extract_dir, h5_files = stage_zip_and_extract(src_zip, LOCAL_STAGE_ROOT)
        orbit  = _orbit_key(src_zip)
        tc_zip = tc_zip_by_orbit.get(orbit)
        tc_h5_by_orbit = {}
        if tc_zip is not None:
            local_tc_zip, tc_extract_dir, tc_h5_files = stage_zip_and_extract(tc_zip, LOCAL_STAGE_TC_ROOT)
            tc_h5_by_orbit = {_orbit_key(p): p for p in tc_h5_files if _orbit_key(p)}
        for fp in h5_files:
            tc_fp = tc_h5_by_orbit.get(_orbit_key(fp))
            merge_acc(test_global, accumulate_one_file(fp, lat_bins, lon_bins, target_h, tc_fp=tc_fp))
            processed_h5_test += 1
    except Exception as ex:
        failed_test.append((str(src_zip), str(ex)))
    finally:
        cleanup_staged_data(local_zip, extract_dir)
        cleanup_staged_data(local_tc_zip, tc_extract_dir)

print(f"Tested: {N_QUICK_TEST} ZIPs | {processed_h5_test} H5 files | {len(failed_test)} failures")
for v in DATA_VARS:
    c = test_global[v]["count"]
    s = test_global[v]["sum"]
    mean_all = np.divide(s, c, out=np.full_like(s, np.nan), where=c > 0)
    print(f"  {SHORT_NAMES[v]:15s}: valid cells={np.isfinite(mean_all).sum()}  "
          f"mean={np.nanmean(mean_all):.3g}")
if failed_test:
    print("First failure:", failed_test[0][1])

**7. Full Batch Processing**

In [ ]:
# Re-initialise (safe to re-run)
global_acc = {
    v: {k: np.zeros((n_lat, n_lon, n_height), dtype=np.float64)
        for k in ("sum", "sum_sq", "count", "sum_err", "count_err")}
    for v in DATA_VARS
}

failed_zips  = []
processed_h5 = 0
n_zips       = len(zip_paths)

for idx, src_zip in enumerate(zip_paths, start=1):
    local_zip = local_tc_zip = None
    extract_dir = tc_extract_dir = None
    try:
        local_zip, extract_dir, h5_files = stage_zip_and_extract(src_zip, LOCAL_STAGE_ROOT)
        orbit  = _orbit_key(src_zip)
        tc_zip = tc_zip_by_orbit.get(orbit)
        tc_h5_by_orbit = {}
        if tc_zip is not None:
            local_tc_zip, tc_extract_dir, tc_h5_files = stage_zip_and_extract(tc_zip, LOCAL_STAGE_TC_ROOT)
            tc_h5_by_orbit = {_orbit_key(p): p for p in tc_h5_files if _orbit_key(p)}
        for fp in h5_files:
            tc_fp = tc_h5_by_orbit.get(_orbit_key(fp))
            merge_acc(global_acc, accumulate_one_file(fp, lat_bins, lon_bins, target_h, tc_fp=tc_fp))
            processed_h5 += 1
    except Exception as e:
        failed_zips.append((str(src_zip), str(e)))
    finally:
        cleanup_staged_data(local_zip, extract_dir)
        cleanup_staged_data(local_tc_zip, tc_extract_dir)
    if idx % 10 == 0 or idx == n_zips:
        print(f"{idx}/{n_zips} ZIPs | h5={processed_h5} | failed={len(failed_zips)}")

print("\nBatch done")
print(f"H5 files: {processed_h5}  |  ZIP failures: {len(failed_zips)}")
if failed_zips:
    print("\nFirst 3 failures:")
    for z, err in failed_zips[:3]:
        print(" -", z, "\n   ", err)

**8. Compute Statistics (mean, std dev, mean error)**

In [ ]:
def compute_statistics(global_acc):
    """
    Derive mean, std dev, and mean retrieval error from accumulated sums.
    Cells with < MIN_SAMPLES_PER_CELL valid observations are masked to NaN.
    """
    stats = {}
    for v in DATA_VARS:
        c   = global_acc[v]["count"]
        s   = global_acc[v]["sum"]
        s2  = global_acc[v]["sum_sq"]
        ce  = global_acc[v]["count_err"]
        se  = global_acc[v]["sum_err"]

        mean    = np.divide(s,  c,  out=np.full_like(c,  np.nan, dtype=float), where=c  > 0)
        mean_sq = np.divide(s2, c,  out=np.full_like(c,  np.nan, dtype=float), where=c  > 0)
        variance = mean_sq - mean**2
        std = np.sqrt(np.maximum(variance, 0.0))
        std[c <= 1] = np.nan   # undefined for n=0 or n=1

        mean_err = np.divide(se, ce, out=np.full_like(ce, np.nan, dtype=float), where=ce > 0)

        # Mask sparse cells
        if MIN_SAMPLES_PER_CELL and MIN_SAMPLES_PER_CELL > 0:
            mean[c     < MIN_SAMPLES_PER_CELL] = np.nan
            std[c      < MIN_SAMPLES_PER_CELL] = np.nan
            mean_err[c < MIN_SAMPLES_PER_CELL] = np.nan

        stats[v] = {"mean": mean, "std": std, "mean_err": mean_err, "count": c}
    return stats


stats = compute_statistics(global_acc)

print("Statistics computed")
print(f"{'variable':20s}  {'valid cells':>12s}  {'global mean':>12s}")
for v in DATA_VARS:
    m = stats[v]["mean"]
    print(f"  {SHORT_NAMES[v]:18s}  {np.isfinite(m).sum():12d}  {np.nanmean(m):12.4g}")

**9. Save to NetCDF**

In [ ]:
outdir = f"{OUTPUT_ROOT}/{start_dt:%Y%m%d}_{end_dt:%Y%m%d}"
os.makedirs(outdir, exist_ok=True)
_base = f"ATL_EBD_2A_{GRID_RES_DEG}deg_{start_dt:%Y%m%d}_{end_dt:%Y%m%d}"

# --- 3D file: (latitude, longitude, height) ---
ds_3d = xr.Dataset(coords={"latitude": lat_centers, "longitude": lon_centers, "height": target_h})
for v in DATA_VARS:
    sn = SHORT_NAMES[v]
    dims = ["latitude", "longitude", "height"]
    ds_3d[f"{sn}_mean"]     = (dims, stats[v]["mean"])
    ds_3d[f"{sn}_std"]      = (dims, stats[v]["std"])
    ds_3d[f"{sn}_err_mean"] = (dims, stats[v]["mean_err"])
    ds_3d[f"{sn}_count"]    = (dims, stats[v]["count"])
ds_3d.to_netcdf(f"{outdir}/{_base}_3d.nc")

# --- Lat/height file: zonal mean (longitude collapsed) ---
ds_lh = xr.Dataset(coords={"latitude": lat_centers, "height": target_h})
for v in DATA_VARS:
    sn   = SHORT_NAMES[v]
    dims = ["latitude", "height"]
    # Sum across longitude axis (axis=1) to build zonal accumulators
    c_lh  = np.nansum(global_acc[v]["count"],     axis=1)
    s_lh  = np.nansum(global_acc[v]["sum"],       axis=1)
    s2_lh = np.nansum(global_acc[v]["sum_sq"],    axis=1)
    ce_lh = np.nansum(global_acc[v]["count_err"], axis=1)
    se_lh = np.nansum(global_acc[v]["sum_err"],   axis=1)

    mean_lh    = np.divide(s_lh,  c_lh,  out=np.full_like(c_lh,  np.nan), where=c_lh  > 0)
    mean_sq_lh = np.divide(s2_lh, c_lh,  out=np.full_like(c_lh,  np.nan), where=c_lh  > 0)
    std_lh = np.sqrt(np.maximum(mean_sq_lh - mean_lh**2, 0.0))
    std_lh[c_lh <= 1] = np.nan
    mean_err_lh = np.divide(se_lh, ce_lh, out=np.full_like(ce_lh, np.nan), where=ce_lh > 0)

    if MIN_SAMPLES_PER_CELL and MIN_SAMPLES_PER_CELL > 0:
        mean_lh[c_lh     < MIN_SAMPLES_PER_CELL] = np.nan
        std_lh[c_lh      < MIN_SAMPLES_PER_CELL] = np.nan
        mean_err_lh[c_lh < MIN_SAMPLES_PER_CELL] = np.nan

    ds_lh[f"{sn}_mean"]     = (dims, mean_lh)
    ds_lh[f"{sn}_std"]      = (dims, std_lh)
    ds_lh[f"{sn}_err_mean"] = (dims, mean_err_lh)
ds_lh.to_netcdf(f"{outdir}/{_base}_latheight.nc")

print(f"Saved to {outdir}/")
print(f"  {_base}_3d.nc         (lat x lon x height, all variables)")
print(f"  {_base}_latheight.nc  (lat x height, zonal mean)")